# 전이학습과 미세조정 실습

**Transfer Learning · Fine-tuning · 미세조정**

한 과제에서 학습한 모델을 데이터가 적은 다른 과제에 옮겨 다시 학습시키는 방법.

소재 분야에서 이해하기: 대규모 DFT 데이터로 학습한 모델을 소수의 실험값으로 조정한다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [Google ML 용어집](https://developers.google.com/machine-learning/glossary)

## 1. 얼마나 적은 데이터로 옮길 수 있나

미세조정에 쓰는 데이터 수를 바꿔가며 전이학습의 이득을 확인합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

import copy
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error

def task(shift, n, seed):
    local = np.random.default_rng(seed)
    x = local.uniform(0, 1, (n, 3))
    y = np.sin(3 * x[:, 0]) + 2 * x[:, 1] ** 2 - x[:, 2] + shift + local.normal(0, 0.05, n)
    return x, y

X_big, y_big = task(0.0, 4000, 1)
X_eval, y_eval = task(0.4, 500, 3)
base = MLPRegressor(hidden_layer_sizes=(64, 64), max_iter=3000, random_state=0).fit(X_big, y_big)

rows = []
for n in (5, 10, 25, 50, 100, 300):
    X_few, y_few = task(0.4, n, 10 + n)
    scratch = MLPRegressor(hidden_layer_sizes=(64, 64), max_iter=3000, random_state=0).fit(X_few, y_few)
    tuned = copy.deepcopy(base)
    tuned.set_params(learning_rate_init=0.001)
    for _ in range(40):
        tuned.partial_fit(X_few, y_few)
    rows.append((n, mean_absolute_error(y_eval, scratch.predict(X_eval)),
                 mean_absolute_error(y_eval, tuned.predict(X_eval))))
    print('n=%3d  처음부터 %.4f  전이학습 %.4f' % rows[-1])

rows = np.array(rows)
plt.plot(rows[:, 0], rows[:, 1], 'o-', label='from scratch')
plt.plot(rows[:, 0], rows[:, 2], 's-', label='transfer learning')
plt.xscale('log'); plt.xlabel('target-task samples'); plt.ylabel('test MAE'); plt.legend(); plt.show()

## 2. 해석

데이터가 적을 때 전이학습의 이득이 가장 큽니다. 데이터가 충분해지면 두 방법의 차이는 줄어듭니다.
소재 분야에서 계산 데이터로 사전학습하고 소수의 실험값으로 맞추는 방식이 자주 쓰이는 이유입니다.

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#transfer-learning)을 여세요.